Environment

This notebook is intended to run in the project virtual environment:

Python (ns-health-signals)

All dependencies are listed in requirements.txt.

# 03 - Merge System Signals (Nova Scotia)

This notebook merges the processed system “signal” datasets into a single modeling-ready table.

## Inputs

- Primary care access activity (monthly)  
  `data_processed/primary_care_access_activity_ns.csv`

- EMS demand (EHS responses, monthly)  
  `data_processed/ehs_demand_ns.csv`

## Output

Merged dataset:  
`data_processed/system_signals_merged_ns.csv`

## Why this is important

Combining upstream primary care access activity with downstream EMS demand creates a stronger Nova Scotia health system signal suitable for early-warning analysis and modeling.

In [1]:
from pathlib import Path
import pandas as pd
import os

## Resolve project paths

This ensures the notebook works whether Jupyter is opened from the project root or the notebooks folder.

In [2]:
cwd = Path.cwd()
project_root = cwd if (cwd / "data_processed").exists() else cwd.parent
data_processed_dir = project_root / "data_processed"

pc_path = data_processed_dir / "primary_care_access_activity_ns.csv"
ehs_path = data_processed_dir / "ehs_demand_ns.csv"

print("Working directory:", cwd)
print("Project root:", project_root)
print("Primary care file exists:", pc_path.exists())
print("EHS file exists:", ehs_path.exists())

Working directory: c:\ns-health-signals\notebooks
Project root: c:\ns-health-signals
Primary care file exists: True
EHS file exists: True


## Load processed signal datasets

We load the cleaned outputs from Notebook 01 and Notebook 02 and confirm basic structure.

In [9]:
pc = pd.read_csv(pc_path, parse_dates=["Date"])
ehs = pd.read_csv(ehs_path, parse_dates=["Date"])

print("Primary care shape:", pc.shape)
print("EHS shape:", ehs.shape)

pc.head(), ehs.head()

Primary care shape: (146, 5)
EHS shape: (136, 6)
Index(['Zone', 'Date', 'ehs_responses', 'responses_roll3',
       'responses_pct_change', 'responses_zscore'],
      dtype='object')


(      Zone       Date  primary_care_visits  visits_roll3  visits_pct_change
 0  Central 2020-12-01                589.0    589.000000                NaN
 1  Central 2021-01-01                363.0    476.000000          -0.383701
 2  Central 2021-02-01                559.0    503.666667           0.539945
 3  Central 2021-03-01                823.0    581.666667           0.472272
 4  Central 2021-04-01                926.0    769.333333           0.125152,
       Zone       Date  ehs_responses  responses_roll3  responses_pct_change  \
 0  Central 2021-03-01          344.0       344.000000                   NaN   
 1  Central 2021-04-01         3553.0      1948.500000              9.328488   
 2  Central 2021-05-01         4805.0      2900.666667              0.352378   
 3  Central 2021-06-01         3737.0      4031.666667             -0.222268   
 4  Central 2021-07-01         3766.0      4102.666667              0.007760   
 
    responses_zscore  
 0         -4.493796  
 1       

## Standardize column names

We rename columns so the merged dataset is easy to read and avoids name collisions.

In [10]:
pc = pc.rename(columns={
    "primary_care_visits": "pc_visits",
    "visits_roll3": "pc_visits_roll3",
    "visits_pct_change": "pc_visits_pct_change"
})

ehs = ehs.rename(columns={
    "responses_zscore": "ehs_responses_zscore"
})

print("PC columns:", list(pc.columns))
print("EHS columns:", list(ehs.columns))

PC columns: ['Zone', 'Date', 'pc_visits', 'pc_visits_roll3', 'pc_visits_pct_change']
EHS columns: ['Zone', 'Date', 'ehs_responses', 'responses_roll3', 'responses_pct_change', 'ehs_responses_zscore']


## Merge by Zone and Date

We merge the datasets on Zone and Date (monthly).  
An inner join keeps only zone-months present in both datasets.

In [5]:
merged = pc.merge(ehs, on=["Zone", "Date"], how="inner")

print("Merged shape:", merged.shape)
merged.head()

Merged shape: (132, 9)


,Zone,Date,pc_visits,pc_visits_roll3,pc_visits_pct_change,ehs_responses,ehs_responses_roll3,ehs_responses_pct_change,responses_zscore
0,Central,2021-03-01,823.0,581.666667,0.472272,344.0,344.000000,NaN,-4.493796
1,Central,2021-04-01,926.0,769.333333,0.125152,3553.0,1948.500000,9.328488,-0.443670
2,Central,2021-05-01,534.0,761.000000,-0.423326,4805.0,2900.666667,0.352378,1.136497
3,Central,2021-06-01,800.0,753.333333,0.498127,3737.0,4031.666667,-0.222268,-0.211441
4,Central,2021-07-01,533.0,622.333333,-0.333750,3766.0,4102.666667,0.007760,-0.174840


## Validate merged coverage

We check zones, date range, and missing values to ensure the merge produced a clean modeling table.

In [6]:
print("Zones:", merged["Zone"].unique())
print("Date range:", merged["Date"].min().date(), "to", merged["Date"].max().date())

missing = merged.isna().sum().sort_values(ascending=False)
missing[missing > 0]

Zones: ['Central' 'Eastern' 'Northern' 'Western']
Date range: 2021-03-01 to 2023-12-01


ehs_responses_pct_change    4
dtype: int64

## Handle expected missing values

Percent change features are undefined for the first month in each zone.  
We remove those initial rows to keep the modeling dataset fully numeric.

In [7]:
before = merged.shape[0]

merged = merged.dropna(subset=["ehs_responses_pct_change"]).copy()

after = merged.shape[0]
print(f"Dropped {before - after} rows with NaN percent change (expected first month per zone).")

merged.isna().sum().sort_values(ascending=False)

Dropped 4 rows with NaN percent change (expected first month per zone).


Zone                        0
Date                        0
pc_visits                   0
pc_visits_roll3             0
pc_visits_pct_change        0
ehs_responses               0
ehs_responses_roll3         0
ehs_responses_pct_change    0
responses_zscore            0
dtype: int64

## Save merged dataset for modeling

We export the merged system signals dataset so the modeling notebook can load it directly.

In [8]:
out_path = data_processed_dir / "system_signals_merged_ns.csv"
merged.to_csv(out_path, index=False)

print("Saved:", out_path)
print("Exists:", out_path.exists())
print("Files in data_processed:", sorted(os.listdir(data_processed_dir)))

Saved: c:\ns-health-signals\data_processed\system_signals_merged_ns.csv
Exists: True
Files in data_processed: ['ehs_demand_ns.csv', 'primary_care_access_activity_ns.csv', 'system_signals_merged_ns.csv']
